## Context

This notebook accompanies User Guide section 16 (Governed Tool Gateway). It builds a `GovernedToolGateway`, then exercises each of the three gates — schema, policy, plausibility — independently, so the reader can see exactly which gate caught which call. The closing visualization is a `(call, gate)` decision grid that summarizes the full transcript at a glance.

In [ ]:
# Phase 1B preamble: bootstrap runtime + load shared helpers
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# knowlytix/harness/governance/tutorials/notebooks/_shared.py. Works in
# both the source clone (knowlytix/gmsh) and any installed-wheel context
# (notebooks ship inside knowlytix-harness as of the 2026-05-07 move).
import os  # noqa: F401  (some notebooks use os.environ / os.makedirs)
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_marker = Path('knowlytix') / 'harness' / 'governance' / 'tutorials' / 'notebooks' / '_shared.py'
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / _marker).exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from knowlytix.harness.governance.tutorials.notebooks import _helpers_phase1 as h1
h1.bootstrap_runtime()  # stubs modeva if absent
from knowlytix.harness.governance.tutorials.notebooks import _shared as ug
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')
from knowlytix.harness.governance.tutorials.notebooks import _helpers_phase4a as h4a


### Concept: three gates, one transcript

Every call to `gateway.execute(tool, args, context)` runs the three gates in sequence and short-circuits on the first failure. Gate 1 is **schema validation**, which checks that every `required` parameter is present and type-correct against the tool's `input_schema`. Gate 2 is the **policy engine**, which delegates to `PolicyEngine.check` and applies declarative rules (allow/block/requires) plus the four internal layers described in §18. Gate 3 is the **plausibility check**, which scores the geodesic distance $d(R_r v_{\text{ctx}}, v_{\text{tool}})$ on $S^{m-1}$ for the configurable relation `plausibility_relation` (default `should_call`) and hard-blocks if the distance exceeds $\theta_{\text{plaus}}$.

The audit value is uniformity. A `GatewayResult` always carries `gate_results` (per-gate pass/fail), `policy_decision` (full Layer-1..4 verdict from gate 2), `plausibility_score` (geodesic from gate 3), and `reason` (one-line summary). A reviewer reading the transcript does not need to know which tool ran — the shape is identical across the catalog. That uniformity is what makes downstream verifier, tracer, and SPC-gate logic composable.

Below we trigger each gate by construction. We omit a required parameter to fail gate 1; we register a custom policy rule to fail gate 2; and we tighten `theta_plausibility` and inject a deterministic geodesic to fail gate 3. The final cell builds the decision grid and asserts that each gate caught at least one call.

In [ ]:
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.governed_gateway import (
    GovernedToolGateway,
)
from knowlytix.harness.governance.tools import TOOL_DEFINITIONS
from knowlytix.harness.testing import PolicyEngine, PolicyRule

store   = ug.load_ds_practices_store()
session = SessionState()
policy  = PolicyEngine(store=store)
gateway = GovernedToolGateway(
    session=session,
    store=store,
    policy=policy,
    theta_plausibility=1.5,
    tool_schemas=TOOL_DEFINITIONS,
    device=str(DEVICE),
)
print('gateway ready. theta =', gateway.theta_plausibility)
print('policy rules registered:', len(policy._rules))


### Demo 1: schema gate fail

We call `load_dataset` without the required `name` parameter. Gate 1 short-circuits before the policy or plausibility gates fire.

In [ ]:
# 'name' is the only required parameter for load_dataset.
# Omitting it must trigger gate 1.
r1 = gateway.execute(
    'load_dataset',
    {'builtin': 'iris'},   # missing required 'name'
    context='data_loading',
)
print(f'blocked = {r1.blocked}')
print(f'reason  = {r1.reason}')
print(f'gates   = {r1.gate_results}')


### Demo 2: policy gate fail (custom rule)

We register a `PolicyRule` that hard-blocks `train_model` via the `blocks` rule type, then call `train_model` with fully valid arguments. Gate 1 passes; gate 2 fires.

In [ ]:
# Register a policy rule that forbids train_model.
rule = PolicyRule(
    rule_id='rule-no-train',
    action='train_model',
    rule_type='blocks',
    requirement='compliance_freeze',
    context='*',
    severity='error',
)
gateway.add_policy_rule(rule)
print('rules now registered:', len(gateway.policy._rules))

r2 = gateway.execute(
    'train_model',
    {'dataset': 'iris',
     'model_type': 'RandomForestClassifier',
     'model_name': 'rf_iris',
     'params': {'n_estimators': 10, 'random_state': 0}},
    context='model_training',
)
print(f'blocked = {r2.blocked}')
print(f'reason  = {r2.reason}')
print(f'gates   = {r2.gate_results}')
print(f'blocking_rules = {r2.policy_decision.blocking_rules}')


### Demo 3: plausibility gate fail

Gate 3 scores `(context, plausibility_relation, tool_name)` on the GMS manifold. The `ds_practices_store` fixture does not include a `should_call` relation, so the production score returns None and gate 3 fails open. To exercise the *blocking* path in a tutorial we subclass the gateway and inject a deterministic high geodesic, then tighten `theta_plausibility` below it. In production, the same block fires when the policy graph contains an explicit `(context, should_call, tool)` edge whose distance exceeds the calibrated threshold (see §18 and §24).

In [ ]:
# Subclass to force a deterministic high geodesic so gate 3
# blocks. In production, _score_plausibility returns the
# real distance from store.score_triple(context,
# plausibility_relation, tool_name).
class TightGateway(GovernedToolGateway):
    def _score_plausibility(self, context, tool_name):
        return 1.95

tight = TightGateway(
    session=SessionState(),
    store=store,
    policy=PolicyEngine(store=store),
    theta_plausibility=0.5,
    tool_schemas=TOOL_DEFINITIONS,
    device=str(DEVICE),
)
r3 = tight.execute(
    'load_dataset',
    {'name': 'iris', 'builtin': 'iris'},
    context='unfamiliar_context',
)
print(f'blocked = {r3.blocked}')
print(f'reason  = {r3.reason}')
print(f'gates   = {r3.gate_results}')
print(f'plausibility_score = {r3.plausibility_score}')


### Demo 4: a clean call for contrast

A clean call against the original `gateway` (no custom rule for `data_summary`, default $\theta=1.5$) passes all three gates. We add it to the trace so the grid below has at least one row of greens.

In [ ]:
# Note: a 'data_summary' call can fail with a tool-side
# error if the dataset has not actually been loaded into
# the session. Whether the call returns a non-empty
# display string is irrelevant for this notebook --
# we only care that all three gates passed (blocked is
# False), so the call counts toward the per-gate audit
# trail.
r4 = gateway.execute(
    'data_summary',
    {'dataset': 'iris'},
    context='exploratory_data_analysis',
)
print(f'blocked = {r4.blocked}')
print(f'reason  = {r4.reason}')
print(f'gates   = {r4.gate_results}')


In [ ]:
# Aggregate the four calls into a labeled trace.
trace = [
    ('1. schema fail (no name)',       r1.gate_results),
    ('2. policy fail (custom rule)',   r2.gate_results),
    ('3. plausibility fail (theta)',   r3.gate_results),
    ('4. clean call (all pass)',       r4.gate_results),
]
for label, gr in trace:
    print(f'  {label:36s}  {gr}')


In [ ]:
# Visualization: gate-decision flow trace.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.4))
h4a.plot_gate_decision_grid(
    ax, trace,
    title='GovernedToolGateway: per-call gate verdicts',
)
fig.tight_layout()
plt.show()


### Exercises

1. Add a fifth call that passes the wrong type for `name` (e.g., `name=42`). Confirm gate 1 catches it with a 'expected type' reason.
2. Replace the `blocks` rule above with a `requires` rule whose requirement is `human_approval`. With no approval triple in context, gate 2 should still hard-block. With an `allows` rule of higher priority, gate 2 should short-circuit to PASS.
3. Read `gateway.transcript_summary()` and identify the per-gate block counts. Which gate has the highest block rate in this notebook?

### What you should now be able to do

- Build a `GovernedToolGateway` with a custom `PolicyEngine` and a tightened `theta_plausibility`.
- Trigger each of the three gates in isolation.
- Read `result.gate_results`, `result.reason`, and `result.policy_decision` to determine which gate caught a call and why.
- Compose a custom gateway by adding rules to its `PolicyEngine` and (in production) a `should_call` graph to its store.

### Cross-references

- USER_GUIDE.md section 16 (Governed Tool Gateway)
- USER_GUIDE.md section 18 (Policy Engine + Calibrated Policy Triple) — the full surface behind gate 2
- `knowlytix/harness/governance/governed_gateway.py:GovernedToolGateway.execute` — the three-gate pipeline
- `gms_agent_builder_ext/chapters/ch07.tex` — chapter form of the calibrated gateway

In [ ]:
# Final assertion: each of the three gates caught at
# least one call in this notebook.
n_caught_by_schema = sum(
    1 for _, gr in trace
    if gr.get('schema') is False
)
n_caught_by_policy = sum(
    1 for _, gr in trace
    if gr.get('policy') is False
)
n_caught_by_plausibility = sum(
    1 for _, gr in trace
    if gr.get('plausibility') is False
)
print(f'  schema       caught: {n_caught_by_schema}')
print(f'  policy       caught: {n_caught_by_policy}')
print(f'  plausibility caught: {n_caught_by_plausibility}')
assert (n_caught_by_schema >= 1
        and n_caught_by_policy >= 1
        and n_caught_by_plausibility >= 1), (
    'each gate must catch at least one call; got '
    f'schema={n_caught_by_schema}, '
    f'policy={n_caught_by_policy}, '
    f'plausibility={n_caught_by_plausibility}'
)
print('OK: all three gates fired at least once.')
